# 02 - Feature Engineering
**Objective:** Construct economically meaningful features from World Bank and WTO cleaned datasets.
**Features:**
1. Global Demand Index (WTO sector-level)
2. Demand Growth Rate (year-over-year)
3. Rolling 3-year average demand
4. Product Diversification Metrics (HHI, Shannon Entropy)
5. Economic Context: lagged GDP growth, trade volume proxy, import volume proxy, computed GDP per capita
**Input:** `data/processed/01_*.csv` | **Output:** `data/processed/02_*.csv`

In [ ]:
import os, numpy as np, pandas as pd, warnings
from pathlib import Path
warnings.filterwarnings('ignore')
pd.set_option('display.max_columns', 50)
PROJECT_ROOT = Path.cwd().parent.parent
PROCESSED_DIR = PROJECT_ROOT / 'data' / 'processed'
df_wb = pd.read_csv(PROCESSED_DIR / '01_worldbank_cleaned.csv')
df_wto = pd.read_csv(PROCESSED_DIR / '01_wto_cleaned.csv')
print('World Bank shape:', df_wb.shape)
print('WTO shape:', df_wto.shape)

## 2.1 - WTO Demand Features

In [ ]:
df_wto['year'] = pd.to_numeric(df_wto['year'], errors='coerce')
df_wto['value'] = pd.to_numeric(df_wto['value'], errors='coerce')
if 'indicator' in df_wto.columns:
    print('WTO indicators:')
    print(df_wto['indicator'].value_counts().head(10))
if 'product_sector' in df_wto.columns:
    wto_agg = df_wto.groupby(['year', 'product_sector'])['value'].sum().reset_index()
    yearly_max = wto_agg.groupby('year')['value'].transform('max')
    wto_agg['global_demand_index'] = wto_agg['value'] / yearly_max.replace(0, np.nan)
    wto_agg = wto_agg.sort_values(['product_sector', 'year']).copy()
    wto_agg['demand_growth_rate_pct'] = wto_agg.groupby('product_sector')['value'].pct_change() * 100
    wto_agg['demand_3yr_ma'] = wto_agg.groupby('product_sector')['value'].transform(lambda x: x.rolling(window=3, min_periods=1).mean())
    print('WTO demand features computed. Shape:', wto_agg.shape)
    display(wto_agg.head(10))
else:
    wto_agg = pd.DataFrame()
    print('No product_sector column found in WTO data.')

## 2.2 - Product Diversification Metrics (WTO)

In [ ]:
if not df_wto.empty and 'product_sector' in df_wto.columns:
    wto_yearly = df_wto.groupby(['year', 'product_sector'])['value'].sum().reset_index()
    yearly_totals = wto_yearly.groupby('year')['value'].transform('sum')
    wto_yearly['share'] = wto_yearly['value'] / yearly_totals.replace(0, np.nan)
    hhi = wto_yearly.groupby('year').apply(lambda x: (x['share'] ** 2).sum()).reset_index(name='hhi')
    def shannon_entropy(shares):
        shares = shares[shares > 0]
        return -(shares * np.log(shares)).sum()
    entropy = wto_yearly.groupby('year').apply(lambda x: shannon_entropy(x['share'])).reset_index(name='shannon_entropy')
    diversification = hhi.merge(entropy, on='year')
    max_entropy = np.log(wto_yearly['product_sector'].nunique()) if wto_yearly['product_sector'].nunique() > 0 else 1
    diversification['shannon_entropy_norm'] = diversification['shannon_entropy'] / max_entropy
    print('Diversification metrics per year:')
    display(diversification)
else:
    diversification = pd.DataFrame()
    print('Skipping diversification.')

## 2.3 - Economic Context Features (World Bank)

In [ ]:
df_wb['year'] = pd.to_numeric(df_wb['year'], errors='coerce')
indicator_cols = ['NY.GDP.MKTP.CD', 'NY.GDP.MKTP.KD.ZG', 'NY.GDP.PCAP.CD', 'SP.POP.TOTL', 'TG.VAL.TOTL.GD.ZS', 'NE.IMP.GNFS.ZS']
for col in indicator_cols:
    if col in df_wb.columns:
        df_wb[col] = pd.to_numeric(df_wb[col], errors='coerce')
growth_col = 'NY.GDP.MKTP.KD.ZG'
if growth_col in df_wb.columns:
    df_wb = df_wb.sort_values(['country_code', 'year']).copy()
    df_wb['gdp_growth_lag1'] = df_wb.groupby('country_code')[growth_col].shift(1)
trade_col = 'TG.VAL.TOTL.GD.ZS'
gdp_col = 'NY.GDP.MKTP.CD'
if trade_col in df_wb.columns and gdp_col in df_wb.columns:
    df_wb['trade_volume_proxy'] = df_wb[trade_col] * df_wb[gdp_col] / 100
imp_col = 'NE.IMP.GNFS.ZS'
if imp_col in df_wb.columns and gdp_col in df_wb.columns:
    df_wb['import_volume_proxy'] = df_wb[imp_col] * df_wb[gdp_col] / 100
pop_col = 'SP.POP.TOTL'
if gdp_col in df_wb.columns and pop_col in df_wb.columns:
    df_wb['gdp_per_capita_computed'] = df_wb[gdp_col] / df_wb[pop_col].replace(0, np.nan)
print('Economic context features added. Sample:')
display(df_wb.head())
print('New columns:', [c for c in df_wb.columns if c not in indicator_cols + ['country', 'country_code', 'year']])

## 2.4 - Save Feature-Engineered Datasets

In [ ]:
if not wto_agg.empty:
    wto_agg.to_csv(PROCESSED_DIR / '02_wto_demand_features.csv', index=False)
    print('Saved: 02_wto_demand_features.csv')
if not diversification.empty:
    diversification.to_csv(PROCESSED_DIR / '02_diversification_metrics.csv', index=False)
    print('Saved: 02_diversification_metrics.csv')
df_wb.to_csv(PROCESSED_DIR / '02_worldbank_features.csv', index=False)
print('Saved: 02_worldbank_features.csv')
print('\nFeature engineering complete. Proceed to 03_scaling_and_normalization.ipynb.')